In [1]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
import os
import gzip
import shutil
from scipy.ndimage import binary_erosion
from scipy.stats import wilcoxon, mannwhitneyu, shapiro, pearsonr, spearmanr, linregress
from scipy.spatial import KDTree
from sklearn.metrics import roc_curve, auc
import seaborn as sns
from scipy.ndimage import label
from statsmodels.stats.contingency_tables import mcnemar
from matplotlib.collections import LineCollection

In [12]:
def compute_evaluation_metrics(y1,y):
    tp=np.sum(np.minimum(y1,y))
    fn=np.sum(y1-np.minimum(y1,y))
    fp=np.sum(y-np.minimum(y1,y))
    tn=np.sum(np.ones(y.shape)-np.maximum(y1,y))
    acc=(tp+tn)/(tp+fn+fp+tn)
    sens=tp/(tp+fn)
    spe=tn/(tn+fp)
    pre=tp/(tp+fp)
    return(acc,sens,spe,pre)

def compute_hd95_initial(img,img1,voxel_size):
    if np.max(img)>0.5 and np.max(img1)>0.5:
        ii,jj,kk=np.nonzero(img-binary_erosion(img))
        ii1,jj1,kk1=np.nonzero(img1-binary_erosion(img1))
        dist0=[]
        for i,j,k in zip(ii,jj,kk):
            dist=[]
            for i1,j1,k1 in zip(ii1,jj1,kk1):
                dist.append(((voxel_size[0]*(i-i1))**2+(voxel_size[1]*(j-j1))**2+(voxel_size[2]*(k-k1))**2)**0.5)
            dist0.append(np.min(dist))
        dist1=[]
        for i1,j1,k1 in zip(ii1,jj1,kk1):
            dist=[]
            for i,j,k in zip(ii,jj,kk):
                dist.append(((voxel_size[0]*(i-i1))**2+(voxel_size[1]*(j-j1))**2+(voxel_size[2]*(k-k1)**2))**0.5)
            dist1.append(np.min(dist))
        hd95=np.percentile(dist0+dist1,95)
        return hd95
    else:
        return float('nan')
    
def compute_hd95_3d(img,img1,voxel_size):
    if np.max(img)>0.5 and np.max(img1)>0.5:
        ii,jj,kk=np.nonzero(img-binary_erosion(img))
        ii1,jj1,kk1=np.nonzero(img1-binary_erosion(img1))
        tree = KDTree(np.moveaxis(np.array([voxel_size[0]*ii,voxel_size[1]*jj,voxel_size[2]*kk]),0,1))
        tree1 = KDTree(np.moveaxis(np.array([voxel_size[0]*ii1,voxel_size[1]*jj1,voxel_size[2]*kk1]),0,1))
        dist0=[]
        for i,j,k in zip(ii,jj,kk):
            distance, index = tree1.query(np.array([voxel_size[0]*i,voxel_size[1]*j,voxel_size[2]*k]))
            dist0.append(distance)
        dist1=[]
        for i1,j1,k1 in zip(ii1,jj1,kk1):
            distance, index = tree.query(np.array([voxel_size[0]*i1,voxel_size[1]*j1,voxel_size[2]*k1]))
            dist0.append(distance)
        hd95=np.percentile(dist0+dist1,95)
        return hd95
    else:
        return float('nan')
    
def compute_hd95_2d(img,img1,voxel_size):
    if np.max(img)>0.5 and np.max(img1)>0.5:
        ii,jj=np.nonzero(img-binary_erosion(img))
        ii1,jj1=np.nonzero(img1-binary_erosion(img1))
        tree = KDTree(np.moveaxis(np.array([voxel_size[0]*ii,voxel_size[1]*jj]),0,1))
        tree1 = KDTree(np.moveaxis(np.array([voxel_size[0]*ii1,voxel_size[1]*jj1]),0,1))
        dist0=[]
        for i,j in zip(ii,jj):
            distance, index = tree1.query(np.array([voxel_size[0]*i,voxel_size[1]*j]))
            dist0.append(distance)
        dist1=[]
        for i1,j1 in zip(ii1,jj1):
            distance, index = tree.query(np.array([voxel_size[0]*i1,voxel_size[1]*j1]))
            dist0.append(distance)
        hd95=np.percentile(dist0+dist1,95)
        return hd95
    else:
        return float('nan')
    
def evaluate_segmentation(img,img1,voxel_size):
    tp=np.sum(np.minimum(img,img1))
    fn=np.sum(img1-np.minimum(img,img1))
    fp=np.sum(img-np.minimum(img,img1))
    tn=np.sum(np.ones(img.shape)-np.maximum(img,img1))
    dice=2*tp/(2*tp+fn+fp)
    iou=tp/(tp+fn+fp)
    if len(img.shape)==3:
        hd95=compute_hd95_3d(img,img1,voxel_size)
    if len(img.shape)==2:
        hd95=compute_hd95_2d(img,img1,voxel_size)
    sens=tp/(tp+fn)
    spe=tn/(tn+fp)
    if tp+fp>0:
        pre=tp/(tp+fp)
    else:
        pre=float('nan')
    return(np.array([dice,iou,hd95,sens,spe,pre]))

def compute_lesions(img,img1):
    detected=0
    undetected=0
    fp=0
    structure=np.ones((3,3,3),dtype=int)
    labeled, ncomponents = label(np.array(img1),structure)
    for j in range(1,ncomponents+1):
        img_1=np.array(labeled==j, dtype=int)
        if np.max(np.minimum(img_1,img))>=0.5:
            detected+=1
        else:
            undetected+=1
    labeled, ncomponents = label(np.array(img),structure)
    for j in range(1,ncomponents+1):
        img_1=np.array(labeled==j, dtype=int)
        if np.max(np.minimum(img_1,img1))<0.5:
            fp+=1
    return(detected,undetected,fp)

def get_all_edges(bool_img):
    #Get a list of all edges (where the value changes from True to False) in the 2D boolean image.
    #The returned array edges has he dimension (n, 2, 2).
    #Edge i connects the pixels edges[i, 0, :] and edges[i, 1, :].
    #Note that the indices of a pixel also denote the coordinates of its lower left corner.
    edges = []
    ii, jj = np.nonzero(bool_img)
    for i, j in zip(ii, jj):
        # North
        if j == bool_img.shape[1]-1 or not bool_img[i, j+1]:
            edges.append(np.array([[i, j+1],
                                   [i+1, j+1]]))
        # East
        if i == bool_img.shape[0]-1 or not bool_img[i+1, j]:
            edges.append(np.array([[i+1, j],
                                   [i+1, j+1]]))
        # South
        if j == 0 or not bool_img[i, j-1]:
            edges.append(np.array([[i, j],
                                   [i+1, j]]))
        # West
        if i == 0 or not bool_img[i-1, j]:
            edges.append(np.array([[i, j],
                                   [i, j+1]]))
    if not edges:
        return np.zeros((0, 2, 2))
    else:
        return np.array(edges)
    
def close_loop_edges(edges):
    #Combine the edges defined by 'get_all_edges' to closed loops around objects.
    #If there are multiple disconnected objects a list of closed loops is returned.
    #Note that it's expected that all the edges are part of exactly one loop (but not necessarily the same one).
    loop_list = []
    while edges.size != 0:
        loop = [edges[0, 0], edges[0, 1]]  # Start with first edge
        edges = np.delete(edges, 0, axis=0)
        while edges.size != 0:
            # Get next edge (=edge with common node)
            ij = np.nonzero((edges == loop[-1]).all(axis=2))
            if ij[0].size > 0:
                i = ij[0][0]
                j = ij[1][0]
            else:
                loop.append(loop[0])
                # Uncomment to to make the start of the loop invisible when plotting
                # loop.append(loop[1])
                break
            loop.append(edges[i, (j + 1) % 2, :])
            edges = np.delete(edges, i, axis=0)
        loop_list.append(np.array(loop))
    return loop_list

def plot_outlines(bool_img, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()
    edges = get_all_edges(bool_img=bool_img)
    edges = edges - 0.5  # convert indices to coordinates; TODO adjust according to image extent
    outlines = close_loop_edges(edges=edges)
    cl = LineCollection(outlines, **kwargs)
    ax.add_collection(cl)

def blue_col_img(img,img1):
    img=255*(img-np.min(img))/(np.max(img)-np.min(img))
    img1=(img1-np.min(img1))/(np.max(img1)-np.min(img1))
    img_i=(1-img1)*img
    img_j=255-(1-img1)*(255-img)
    img_3=np.moveaxis(np.array([img_i,img_i,img_j],dtype=int),0,2)
    return img_3

def blue_col_img_1(img,img1):
    img=255*(img-np.min(img))/(np.max(img)-np.min(img))
    img1[img1>5]=5
    img1[img1<0]=0
    img1=img1/5
    img_i=(1-img1)*img
    img_j=255-(1-img1)*(255-img)
    img_3=np.moveaxis(np.array([img_i,img_i,img_j],dtype=int),0,2)
    return img_3

In [17]:
folder_path='C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_4'
files=os.listdir(folder_path)
for i in range(len(files)):
    file_path='{}/{}'.format(folder_path,files[i])
    with gzip.open(file_path,'rb') as f_in:
        with open(file_path[:-3],'wb') as f_out:
            shutil.copyfileobj(f_in,f_out)
    os.remove(file_path)

In [3]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
voxels=np.zeros((3,221,4))
l=-1
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        img=nib.load(file_path).get_fdata()
        img=np.array(img>0.5,dtype=int)
        img_i=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])).get_fdata(),dtype=int)
        img_j=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata(),dtype=int)
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                l+=1
                if df1['label'][j]=='pos':
                    voxels[:,l,0]=1
                else:
                    voxels[:,l,0]=0
                voxels[0,l,1]=np.sum(img)
                voxels[1,l,1]=np.sum(img_i)
                voxels[2,l,1]=np.sum(img_j)
                voxel_size=nib.load(file_path).header.get_zooms()
                voxels[0,l,2]=np.sum(img)*voxel_size[0]*voxel_size[1]*voxel_size[2]
                voxels[1,l,2]=np.sum(img_i)*voxel_size[0]*voxel_size[1]*voxel_size[2]
                voxels[2,l,2]=np.sum(img_j)*voxel_size[0]*voxel_size[1]*voxel_size[2]
                voxels[:,l,3]=k

In [ ]:
i1=voxels[0,np.where(voxels[0,:,0]==0),2][0]*0.001
i2=voxels[0,np.where(voxels[0,:,0]==1),2][0]*0.001
i3=voxels[1,np.where(voxels[0,:,0]==0),2][0]*0.001
i4=voxels[1,np.where(voxels[0,:,0]==1),2][0]*0.001
i5=voxels[2,np.where(voxels[0,:,0]==0),2][0]*0.001
i6=voxels[2,np.where(voxels[0,:,0]==1),2][0]*0.001

plt.boxplot([i1,i2,i3,i4,i5,i6],patch_artist=True,
            boxprops=dict(facecolor='white'),
            medianprops=dict(color='black'),
            positions=[1,2,4,5,7,8])
plt.xticks([1.5,4.5,7.5], ['PET', 'PET/MRI', 'PET/MRI-SC'],fontsize=14)
plt.gca().set_ylabel('Predicted positive volume (cm3)',fontsize=14)
plt.rc('ytick',labelsize=14)
fig=plt.gcf()
fig.savefig('fig_1.png',bbox_inches='tight')

In [ ]:
y1=voxels[0,:,0]
y=np.array(voxels[2,:,2]>1000,dtype=int)
print(np.sum(np.minimum(y,y1)))
print(np.sum(y1-np.minimum(y,y1)))
print(np.sum(y-np.minimum(y,y1)))
print(np.sum(np.ones(y.shape)-np.maximum(y,y1)))

In [ ]:
#0.343 #0.0743,0.0428

In [207]:
wilcoxon(voxels[1,voxels[0,:,0]==0,2],voxels[0,voxels[0,:,0]==0,2])

WilcoxonResult(statistic=788.5, pvalue=0.04286459997763165)

In [191]:
y1=voxels[0,:,0]
l0=np.array(y1==np.array(voxels[1,:,2]>1000,dtype=int),dtype=int)
l1=np.array(y1==np.array(voxels[2,:,2]>1000,dtype=int),dtype=int)
u1=np.sum(np.minimum(l0,l1))
u2=np.sum(l1-np.minimum(l0,l1))
u3=np.sum(l0-np.minimum(l0,l1))
u4=np.sum(np.ones(l0.shape)-np.maximum(l0,l1))
print(mcnemar([[u1,u2],[u3,u4]]))
#0 vs 1: 0.454, 0 vs 2: 0.263, 1 vs 2: 0.831

pvalue      0.8318119049072266
statistic   10.0


In [198]:
eval=np.zeros((5,5))
for k in range(5):
    i0=np.min(np.where(voxels[0,:,3]==k))
    i1=np.max(np.where(voxels[0,:,3]==k))+1
    y1=voxels[0,i0:i1,0]
    y=np.array(voxels[2,i0:i1,2]>1000,dtype=int)
    eval[k,0:4]=compute_evaluation_metrics(y1,y)
    fpr,tpr,thresholds=roc_curve(voxels[0,i0:i1,0],voxels[2,i0:i1,2])
    eval[k,4]=auc(fpr,tpr)

In [199]:
print(np.round(np.mean(eval[:,0]),3),'pmin',np.round(np.std(eval[:,0]),3),'&',np.round(np.mean(eval[:,1]),3),'pmin',np.round(np.std(eval[:,1]),3),'&',np.round(np.mean(eval[:,2]),3),'pmin',np.round(np.std(eval[:,2]),3),'&',np.round(np.mean(eval[:,3]),3),'pmin',np.round(np.std(eval[:,3]),3),'&',np.round(np.mean(eval[:,4]),3),'pmin',np.round(np.std(eval[:,4]),3))

0.814 pmin 0.055 & 0.837 pmin 0.047 & 0.795 pmin 0.136 & 0.802 pmin 0.107 & 0.894 pmin 0.057


In [ ]:
y=voxels[0,:,0]
y1=voxels[0,:,2]
y2=voxels[1,:,2]
y3=voxels[2,:,2]
fpr1, tpr1, thresholds1 = roc_curve(y,y1,drop_intermediate=False)
fpr2, tpr2, thresholds2 = roc_curve(y,y2,drop_intermediate=False)
fpr3, tpr3, thresholds3 = roc_curve(y,y3,drop_intermediate=False)
plt.plot(fpr1, tpr1,color='gray',label='PET')
plt.plot(fpr2, tpr2,color='blue',label='PET/MRI')
plt.plot(fpr3, tpr3,color='black',label='PET/MRI-SC')
plt.rc('ytick',labelsize=14)
plt.rc('xtick',labelsize=16)
plt.legend(loc='lower right',fontsize=13)
plt.ylabel('True positive rate',fontsize=14)
plt.xlabel('False positive rate',fontsize=14)
fig=plt.gcf()
fig.savefig('fig_2.png',bbox_inches='tight')

In [39]:
df=pd.DataFrame(np.transpose(np.array([y,y1,y2,y3])))
df.to_csv('rocs.csv',index=False)

In [78]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
lesions=np.zeros((3,104,4))
fps_for_neg=np.zeros((3,117,2))
structure=np.ones((3,3,3),dtype=int)
l=-1
l111=-1
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        img=nib.load(file_path).get_fdata()
        img=np.array(img>0.5,dtype=int)
        img_i=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])).get_fdata(),dtype=int)
        img_j=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata(),dtype=int)
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    l+=1
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                    img1=nib.load(img1_path).get_fdata()
                    img1=np.array(img1>0.5,dtype=int)
                    lesions[0,l,:3]=compute_lesions(img,img1)
                    lesions[1,l,:3]=compute_lesions(img_i,img1)
                    lesions[2,l,:3]=compute_lesions(img_j,img1)
                    lesions[:,l,3]=k
                else:
                    l111+=1
                    labeled, ncomponents = label(np.array(img),structure)
                    fps_for_neg[0,l111,0]=ncomponents
                    labeled, ncomponents = label(np.array(img_i),structure)
                    fps_for_neg[1,l111,0]=ncomponents
                    labeled, ncomponents = label(np.array(img_j),structure)
                    fps_for_neg[2,l111,0]=ncomponents
                    fps_for_neg[:,l111,1]=k
        print(i)

0
1
2
3
4
5
6
7
8
9


KeyboardInterrupt: 

In [79]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
lll=[]
l=-1
l111=-1
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        img=nib.load(file_path).get_fdata()
        img=np.array(img>0.5,dtype=int)
        img_i=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])).get_fdata(),dtype=int)
        img_j=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata(),dtype=int)
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    l+=1
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                    img1=nib.load(img1_path).get_fdata()
                    img1=np.array(img1>0.5,dtype=int)
                    structure=np.ones((3,3,3),dtype=int)
                    labeled, ncomponents = label(np.array(img1),structure)
                    for j in range(1,ncomponents+1):
                        l1=np.zeros((3))
                        img_1=np.array(labeled==j, dtype=int)
                        if np.max(np.minimum(img_1,img))>=0.5:
                            l1[0]+=1
                        if np.max(np.minimum(img_1,img_i))>=0.5:
                            l1[1]+=1
                        if np.max(np.minimum(img_1,img_j))>=0.5:
                            l1[2]+=1
                        lll.append(l1)
        print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43


In [84]:
lll=np.array(lll)
con=np.zeros((2,2))
for i in range(lll.shape[0]):
    con[int(lll[i,1]),int(lll[i,0])]+=1
print(con)
print(mcnemar(con))
#0.23,

[[ 46.  16.]
 [  9. 135.]]
pvalue      0.2295229434967041
statistic   9.0


In [46]:
i=2
print(np.sum(lesions[i,:,0]))
print(np.sum(lesions[i,:,1]))
print(np.sum(lesions[i,:,2]))
print(np.sum(fps_for_neg[i,:,0]))
print(np.mean(lesions[i,:,0]/(lesions[i,:,0]+lesions[i,:,1])))
print(np.std(lesions[i,:,0]/(lesions[i,:,0]+lesions[i,:,1])))
l0=lesions[i,:,0]+lesions[i,:,2]
print(np.mean(lesions[i,l0>0,0]/l0[l0>0]))
print(np.std(lesions[i,l0>0,0]/l0[l0>0]))

v=[]
v1=[]
for k in range(5):
    i0=np.min(np.where(lesions[0,:,3]==k))
    i1=np.max(np.where(lesions[0,:,3]==k))+1
    y1=voxels[0,i0:i1,0]
    l0=np.sum(lesions[i,i0:i1,0])
    l1=np.sum(lesions[i,i0:i1,0]+lesions[i,i0:i1,1])
    l3=np.sum(lesions[i,i0:i1,0]+lesions[i,i0:i1,2])
    print(i0,i1,l0,l1)
    v.append(l0/l1)
    v1.append(l0[l3>0]/l3[l3>0])
print(np.round(np.mean(v),3),'pmin',np.round(np.std(v),3))
print(np.round(np.mean(v1),3),'pmin',np.round(np.std(v1),3))

148.0
58.0
73.0
77.0
0.7780448717948717
0.36060976029648717
0.7256723074904893
0.34713338526762216
0 21 29.0 34.0
21 42 33.0 38.0
42 63 31.0 41.0
63 84 35.0 60.0
84 104 20.0 33.0
0.733 pmin 0.12
0.664 pmin 0.057


In [ ]:
df=pd.read_excel('C:/Users/Oona/Documents/Tpc/hnc/hnc_types.xlsx')
l=-1
types=[]
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    l+=1
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])                   
                    for i1 in range(df.shape[0]):    
                        if 'anon' in df1['anom_folder'][j]:
                            if df['index'][i1]==int(df1['anom_folder'][j][:-5]):
                                types.append(df['type'][i1])
                                if df['index'][i1] in [5,6,7,8,9,10,13,14]:
                                    print(k,df['index'][i1])
                        else:
                            if df['index'][i1]==int(df1['anom_folder'][j]):
                                types.append(df['type'][i1])
                                if df['index'][i1] in [5,6,7,8,9,10,13,14]:
                                    print(k,df['index'][i1])
                                    
#[5_anon,6_anon], [7_anon,8_anon], [13_anon,14_anon]

#[2,3], [4,5], [6,7], [12,13], [16,17], [18,19,20], [31,32], [36,37], [39,40], [41,42]

0 8
1 6
2 9
3 14
4 13
4 5
4 7


In [6]:
df=pd.read_excel('C:/Users/Oona/Documents/Tpc/hnc/hnc_types.xlsx')
types=[]
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])                   
                    for i1 in range(df.shape[0]):    
                        if 'anon' in df1['anom_folder'][j]:
                            if df['index'][i1]==int(df1['anom_folder'][j][:-5]):
                                types.append(df['type'][i1])
                        else:
                            if df['index'][i1]==int(df1['anom_folder'][j]):
                                types.append(df['type'][i1])

In [48]:
print(np.unique(types))

['hypopharynx' 'larynx' 'nasopharynx' 'non-scc' 'oral' 'oropharynx'
 'other scc' 'unknown primary']


In [58]:
i1=2
for i in range(len(np.unique(types))):
    print(np.unique(types)[i])
    detected=0
    undetected=0
    for j in range(len(types)):
        if types[j]==np.unique(types)[i]:
            detected+=lesions[i1,j,0]
            undetected+=lesions[i1,j,1]
    #print(detected+undetected)
    print(detected)
    print(round(detected/(detected+undetected),3))

hypopharynx
21.0
0.677
larynx
7.0
0.636
nasopharynx
7.0
0.778
non-scc
16.0
0.516
oral
28.0
0.737
oropharynx
48.0
0.787
other scc
6.0
0.75
unknown primary
15.0
0.882


In [ ]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
results=np.zeros((3,104,6))
l=-1
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        img=nib.load(file_path).get_fdata()
        img=np.array(img>0.5,dtype=int)
        img_i=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])).get_fdata()>0.5,dtype=int)
        img_j=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata()>0.5,dtype=int)
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                                    l+=1
                    img1=nib.load(img1_path).get_fdata()
                    img1=np.array(img1>0.5,dtype=int)
                    voxel_size=nib.load(img1_path).header.get_zooms()
                    results[0,l,:]=evaluate_segmentation(img,img1,voxel_size)
                    results[1,l,:]=evaluate_segmentation(img_i,img1,voxel_size)
                    results[2,l,:]=evaluate_segmentation(img_j,img1,voxel_size)
                    print(l)
np.save('results.npy',results)

In [6]:
results=np.load('results.npy')

In [9]:
for i in range(6):
    print(np.round(np.nanmean(results[0,:,i]),3),'pmin',np.round(np.nanstd(results[0,:,i]),3),'&',np.round(np.nanmean(results[1,:,i]),3),'pmin',np.round(np.nanstd(results[1,:,i]),3),'&',np.round(np.nanmean(results[2,:,i]),3),'pmin',np.round(np.nanstd(results[2,:,i]),3))

0.533 pmin 0.311 & 0.584 pmin 0.31 & 0.576 pmin 0.321
0.422 pmin 0.281 & 0.473 pmin 0.279 & 0.469 pmin 0.292
39.715 pmin 49.128 & 36.261 pmin 46.414 & 35.147 pmin 44.649
0.536 pmin 0.346 & 0.588 pmin 0.344 & 0.569 pmin 0.352
1.0 pmin 0.0 & 1.0 pmin 0.0 & 1.0 pmin 0.0
0.708 pmin 0.297 & 0.735 pmin 0.263 & 0.727 pmin 0.294


In [68]:
for i in range(6):
    print(np.round(100*np.nanmean(results[0,:,i]),1),'pmin',np.round(100*np.nanstd(results[0,:,i]),1),'&',np.round(100*np.nanmean(results[1,:,i]),1),'pmin',np.round(100*np.nanstd(results[1,:,i]),1),'&',np.round(100*np.nanmean(results[2,:,i]),1),'pmin',np.round(100*np.nanstd(results[2,:,i]),1))

53.3 pmin 31.1 & 58.4 pmin 31.0 & 57.6 pmin 32.1
42.2 pmin 28.1 & 47.3 pmin 27.9 & 46.9 pmin 29.2
3971.5 pmin 4912.8 & 3626.1 pmin 4641.4 & 3514.7 pmin 4464.9
53.6 pmin 34.6 & 58.8 pmin 34.4 & 56.9 pmin 35.2
100.0 pmin 0.0 & 100.0 pmin 0.0 & 100.0 pmin 0.0
70.8 pmin 29.7 & 73.5 pmin 26.3 & 72.7 pmin 29.4


In [71]:
i=4
print(np.round(100*np.nanmean(results[0,:,i]),3),'pmin',np.round(100*np.nanstd(results[0,:,i]),3),'&',np.round(100*np.nanmean(results[1,:,i]),3),'pmin',np.round(100*np.nanstd(results[1,:,i]),3),'&',np.round(100*np.nanmean(results[2,:,i]),3),'pmin',np.round(100*np.nanstd(results[2,:,i]),3))

99.981 pmin 0.03 & 99.979 pmin 0.035 & 99.982 pmin 0.028


In [74]:
for i in range(6):
    print(wilcoxon(results[0,:,i],results[2,:,i]))

WilcoxonResult(statistic=1495.0, pvalue=0.00815184066644081)
WilcoxonResult(statistic=1443.0, pvalue=0.0044417265440373265)
WilcoxonResult(statistic=nan, pvalue=nan)
WilcoxonResult(statistic=1660.0, pvalue=0.16113180656414516)
WilcoxonResult(statistic=1902.0, pvalue=0.08775477734518804)
WilcoxonResult(statistic=nan, pvalue=nan)


In [ ]:
x=results[0,:,5]
y=results[1,:,5]
l=(np.isnan(x) | np.isnan(y))
wilcoxon(x[l==0],y[l==0])

WilcoxonResult(statistic=1622.0, pvalue=0.11953101544119232)

In [ ]:
i1=results[0,:,0]
i2=results[0,:,1]
i3=results[0,:,3]
i4=results[0,:,4]
i5=results[0,np.isnan(results[0,:,5])==False,5]
i6=results[1,:,0]
i7=results[1,:,1]
i8=results[1,:,3]
i9=results[1,:,4]
i10=results[1,np.isnan(results[1,:,5])==False,5]
i11=results[2,:,0]
i12=results[2,:,1]
i13=results[2,:,3]
i14=results[2,:,4]
i15=results[2,np.isnan(results[2,:,5])==False,5]

fig, ax = plt.subplots()
bp1 = ax.boxplot([i1,i6,i11], positions=[1,6,11], notch=False, widths=0.7, 
                 patch_artist=True, boxprops=dict(facecolor='white'), medianprops=dict(color='black'))
bp2 = ax.boxplot([i2,i7,i12], positions=[2,7,12], notch=False, widths=0.7, 
                 patch_artist=True, boxprops=dict(facecolor='lightblue'), medianprops=dict(color='black'))
bp3 = ax.boxplot([i3,i8,i13], positions=[3,8,13], notch=False, widths=0.7, 
                 patch_artist=True, boxprops=dict(facecolor='deepskyblue'), medianprops=dict(color='black'))
#bp4 = ax.boxplot([i4,i9,i14], positions=[4,10,16], notch=False, widths=0.7, 
#                 patch_artist=True, boxprops=dict(facecolor='cyan'), medianprops=dict(color='black'))
bp5 = ax.boxplot([i5,i10,i15], positions=[4,9,14], notch=False, widths=0.7, 
                 patch_artist=True, boxprops=dict(facecolor='blue'), medianprops=dict(color='black'))

#ax.legend([bp1['boxes'][0], bp2['boxes'][0], bp3['boxes'][0], bp5['boxes'][0]], 
#          ['Dice', 'IoU', 'Sensitivity', 'Precision'], loc='upper right')

ax.set_xlim(0,15)
plt.xticks([2.5,7.5,12.5], ['PET', 'PET/MRI', 'PET/MRI-SC'],fontsize=14)

#plt.legend(loc='lower right')
#plt.gca().set_ylabel('Predicted positive volume (cm^3)')
fig.legend([bp1['boxes'][0], bp2['boxes'][0], bp3['boxes'][0], bp5['boxes'][0]], 
            ['Dice', 'IoU', 'Sensitivity', 'Precision'],bbox_to_anchor=(1.19, 0.9), 
            loc='upper right', fontsize=14, handlelength=1, handleheight=1)
plt.rc('ytick',labelsize=14)
fig=plt.gcf()
fig.savefig('fig_3.png',bbox_inches='tight')
plt.show()

In [153]:
for i in range(len(np.unique(types))):
    dices1=[]
    dices2=[]
    dices3=[]
    for j in range(len(types)):
        if types[j]==np.unique(types)[i]:
            dices1.append(results[0,j,0])
            dices2.append(results[1,j,0])
            dices3.append(results[2,j,0])
    print(np.unique(types)[i],'&',np.round(100*np.mean(dices1),3),'pmin',np.round(100*np.std(dices1),3),'&',np.round(100*np.mean(dices2),3),'pmin',np.round(100*np.std(dices2),3),'&',np.round(100*np.mean(dices3),3),'pmin',np.round(100*np.std(dices3),3))

hypopharynx & 44.302 pmin 29.144 & 44.811 pmin 34.049 & 47.748 pmin 33.276
larynx & 48.114 pmin 39.225 & 55.935 pmin 37.465 & 57.5 pmin 38.855
nasopharynx & 57.026 pmin 27.503 & 57.762 pmin 29.774 & 58.872 pmin 31.099
non-scc & 34.838 pmin 30.128 & 43.186 pmin 30.576 & 44.244 pmin 32.594
oral & 65.692 pmin 26.501 & 64.726 pmin 27.865 & 63.85 pmin 28.473
oropharynx & 48.116 pmin 29.115 & 55.647 pmin 31.591 & 54.796 pmin 31.277
other scc & 50.165 pmin 30.699 & 68.856 pmin 14.435 & 49.666 pmin 34.053
unknown primary & 74.356 pmin 16.072 & 77.774 pmin 13.667 & 78.073 pmin 13.88


In [159]:
for i in range(len(np.unique(types))):
    print(np.unique(types)[i])
    dices1=[]
    dices2=[]
    dices3=[]
    for j in range(len(types)):
        if types[j]==np.unique(types)[i]:
            dices1.append(results[0,j,0])
            dices2.append(results[1,j,0])
            dices3.append(results[2,j,0])
    print(wilcoxon(dices1,dices3))

hypopharynx
WilcoxonResult(statistic=10.0, pvalue=0.1386406338132186)
larynx
WilcoxonResult(statistic=1.0, pvalue=0.027991815485665747)
nasopharynx
WilcoxonResult(statistic=7.0, pvalue=0.463071015014588)
non-scc
WilcoxonResult(statistic=25.0, pvalue=0.47690655496758394)
oral
WilcoxonResult(statistic=83.0, pvalue=0.9133009306692093)
oropharynx
WilcoxonResult(statistic=122.0, pvalue=0.6265140021742197)
other scc
WilcoxonResult(statistic=7.0, pvalue=0.5625)
unknown primary
WilcoxonResult(statistic=23.0, pvalue=0.127197265625)


c:\Users\Oona\AppData\Local\Programs\Python\Python39\lib\site-packages\scipy\stats\_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


In [39]:
sequences=[]
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    sequences.append(df1['mri_sequence'][j])

In [40]:
np.unique(sequences)

array(['spir', 't1c', 't1w', 't1w_tse', 't2', 't2w', 't2w_tse', 'tse',
       'unknown'], dtype='<U7')

In [41]:
t1_s=['spir','t1c','t1w','t1w_tse','tse','unknown']
dices1=[]
dices2=[]
dices3=[]
dices4=[]
dices5=[]
dices6=[]
for j in range(len(sequences)):
    if sequences[j] in t1_s:
        dices1.append(results[0,j,0])
        dices2.append(results[1,j,0])
        dices3.append(results[2,j,0])
    else:
        dices4.append(results[0,j,0])
        dices5.append(results[1,j,0])
        dices6.append(results[2,j,0])
print(len(dices1),'&',np.round(100*np.mean(dices1),3),'pmin',np.round(100*np.std(dices1),3),'&',np.round(100*np.mean(dices2),3),'pmin',np.round(100*np.std(dices2),3),'&',np.round(100*np.mean(dices3),3),'pmin',np.round(100*np.std(dices3),3))
print(len(dices4),'&',np.round(100*np.mean(dices4),3),'pmin',np.round(100*np.std(dices4),3),'&',np.round(100*np.mean(dices5),3),'pmin',np.round(100*np.std(dices5),3),'&',np.round(100*np.mean(dices6),3),'pmin',np.round(100*np.std(dices6),3))

31 & 52.274 pmin 37.25 & 53.884 pmin 35.955 & 52.68 pmin 37.424
73 & 53.733 pmin 28.03 & 60.323 pmin 28.381 & 59.691 pmin 29.293


In [43]:
print(wilcoxon(dices1,dices2))
print(wilcoxon(dices1,dices3))
print(wilcoxon(dices2,dices3))
print(wilcoxon(dices4,dices5))
print(wilcoxon(dices4,dices6))
print(wilcoxon(dices5,dices6))
print(mannwhitneyu(dices1,dices4))
print(mannwhitneyu(dices2,dices5))
print(mannwhitneyu(dices3,dices6))

WilcoxonResult(statistic=120.0, pvalue=0.584056453701191)
WilcoxonResult(statistic=82.0, pvalue=0.052033421360799366)
WilcoxonResult(statistic=123.0, pvalue=0.44045294529422474)
WilcoxonResult(statistic=705.0, pvalue=0.0026608725876412025)
WilcoxonResult(statistic=857.0, pvalue=0.036115852316199085)
WilcoxonResult(statistic=1044.0, pvalue=0.5528925147940282)
MannwhitneyuResult(statistic=1183.0, pvalue=0.7166255486123823)
MannwhitneyuResult(statistic=1063.0, pvalue=0.628412045286372)
MannwhitneyuResult(statistic=1088.5, pvalue=0.7623532386235723)


In [ ]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
volumes=np.zeros((4,104))
l=-1
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        img=nib.load(file_path).get_fdata()
        img=np.array(img>0.5,dtype=int)
        img_i=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])).get_fdata()>0.5,dtype=int)
        img_j=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata()>0.5,dtype=int)
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                                    l+=1
                    img1=nib.load(img1_path).get_fdata()
                    img1=np.array(img1>0.5,dtype=int)
                    voxel_size=nib.load(img1_path).header.get_zooms()
                    volumes[0,l]=np.sum(img1)*voxel_size[0]*voxel_size[1]*voxel_size[2]*0.001
                    volumes[1,l]=np.sum(img)*voxel_size[0]*voxel_size[1]*voxel_size[2]*0.001
                    volumes[2,l]=np.sum(img_i)*voxel_size[0]*voxel_size[1]*voxel_size[2]*0.001
                    volumes[3,l]=np.sum(img_j)*voxel_size[0]*voxel_size[1]*voxel_size[2]*0.001
                    print(l)

In [37]:
print(spearmanr(volumes[0,:],volumes[1,:]))
print(spearmanr(volumes[0,:],volumes[2,:]))
print(spearmanr(volumes[0,:],volumes[3,:]))
print(np.mean(abs(volumes[0,:]-volumes[1,:])))
print(np.mean(abs(volumes[0,:]-volumes[2,:])))
print(np.mean(abs(volumes[0,:]-volumes[3,:])))
print(np.mean(abs(volumes[0,:]-volumes[1,:])/volumes[0,:])*100)
print(np.mean(abs(volumes[0,:]-volumes[2,:])/volumes[0,:])*100)
print(np.mean(abs(volumes[0,:]-volumes[3,:])/volumes[0,:])*100)
print(wilcoxon(abs(volumes[0,:]-volumes[1,:]),abs(volumes[0,:]-volumes[2,:])))
print(wilcoxon(abs(volumes[0,:]-volumes[1,:]),abs(volumes[0,:]-volumes[3,:])))
print(wilcoxon(abs(volumes[0,:]-volumes[2,:]),abs(volumes[0,:]-volumes[3,:])))
print(wilcoxon(abs(volumes[0,:]-volumes[1,:])/volumes[0,:],abs(volumes[0,:]-volumes[2,:])/volumes[0,:]))
print(wilcoxon(abs(volumes[0,:]-volumes[1,:])/volumes[0,:],abs(volumes[0,:]-volumes[3,:])/volumes[0,:]))
print(wilcoxon(abs(volumes[0,:]-volumes[2,:])/volumes[0,:],abs(volumes[0,:]-volumes[3,:])/volumes[0,:]))

SignificanceResult(statistic=0.7459985571311204, pvalue=1.0267227861222987e-19)
SignificanceResult(statistic=0.7994877677650926, pvalue=2.589495761895797e-24)
SignificanceResult(statistic=0.830386770986906, pvalue=1.168332816303948e-27)
9.289681183342509
8.349458338512667
7.720846476352794
57.47469381374904
46.480720523894185
45.86612369361829
WilcoxonResult(statistic=2179.0, pvalue=0.30154467491687253)
WilcoxonResult(statistic=2235.0, pvalue=0.144996621633759)
WilcoxonResult(statistic=2266.0, pvalue=0.2944204784323825)
WilcoxonResult(statistic=1811.0, pvalue=0.02047365901546041)
WilcoxonResult(statistic=1964.0, pvalue=0.018823808064477983)
WilcoxonResult(statistic=2246.0, pvalue=0.26432528269400524)


In [81]:
print(pearsonr(volumes[0,:],volumes[1,:])[0]**2)
print(pearsonr(volumes[0,:],volumes[2,:])[0]**2)
print(pearsonr(volumes[0,:],volumes[3,:])[0]**2)

0.5488720626205312
0.6157013912041502
0.6993801977331632


In [ ]:
#plt.scatter(volumes[0,np.array(types)=='hypopharynx'],volumes[i1,np.array(types)=='hypopharynx'],color='black',marker='*',label='Hypopharynx')
#plt.legend(loc='upper left')
b,a,cor_value,p_value,std_err=linregress(volumes[0,:],volumes[1,:])
print('Slope:',b)
print('Intercept:',a)
plt.scatter(volumes[0,:],volumes[1,:],color='blue')
plt.plot(np.array([-5,155]),b*np.array([-5,155])+a, color='black')
plt.axis('scaled')
plt.xlim([-5,155])
plt.ylim([-5,155])
plt.ylabel('Predicted GTV (cm3)',fontsize=14)
plt.xlabel('True GTV (cm3)',fontsize=14)
plt.title('GTV predicted via PET',fontsize=15)
plt.rc('ytick',labelsize=15)
plt.rc('xtick',labelsize=14)
fig=plt.gcf()
fig.savefig('fig_4.png',bbox_inches='tight')

In [ ]:
b,a,cor_value,p_value,std_err=linregress(volumes[0,:],volumes[2,:])
print('Slope:',b)
print('Intercept:',a)
plt.scatter(volumes[0,:],volumes[2,:],color='blue')
plt.plot(np.array([-5,155]),b*np.array([-5,155])+a, color='black')
plt.axis('scaled')
plt.xlim([-5,155])
plt.ylim([-5,155])
plt.ylabel('Predicted GTV (cm3)',fontsize=14)
plt.xlabel('True GTV (cm3)',fontsize=14)
plt.title('GTV predicted via PET/MRI',fontsize=15)
plt.rc('ytick',labelsize=15)
plt.rc('xtick',labelsize=14)
fig=plt.gcf()
fig.savefig('fig_5.png',bbox_inches='tight')

In [ ]:
b,a,cor_value,p_value,std_err=linregress(volumes[0,:],volumes[3,:])
print('Slope:',b)
print('Intercept:',a)
plt.scatter(volumes[0,:],volumes[3,:],color='blue')
plt.plot(np.array([-5,155]),b*np.array([-5,155])+a, color='black')
plt.axis('scaled')
plt.xlim([-5,155])
plt.ylim([-5,155])
plt.ylabel('Predicted GTV (cm3)',fontsize=14)
plt.xlabel('True GTV (cm3)',fontsize=14)
plt.title('GTV predicted via PET/MRI-SC',fontsize=15)
plt.rc('ytick',labelsize=15)
plt.rc('xtick',labelsize=14)
fig=plt.gcf()
fig.savefig('fig_6.png',bbox_inches='tight')

In [10]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
res_cls=[]
res_seg1=[]
res_seg2=[]
res_seg3=[]
nums=[]
l=-1
for k in range(5):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(len(files)):
        file_path='{}/{}'.format(folder_path,files[i])
        img=nib.load(file_path).get_fdata()
        img_i=nib.load('C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])).get_fdata()
        img_j=nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata()
        img1=np.zeros(img.shape)
        voxel_size=nib.load(file_path).header.get_zooms()
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                                    l+=1
                    img1=nib.load(img1_path).get_fdata()
                    img1=np.array(img1>0.5,dtype=int)
                    voxel_size=nib.load(img1_path).header.get_zooms()
        for j in range(img1.shape[2]):
            res_cls.append([k,i,j,np.max(img1[:,:,j]),np.max(img[:,:,j]),np.max(img_i[:,:,j]),np.max(img_j[:,:,j])])
        img=np.array(img>0.5,dtype=int)
        img_i=np.array(img_i>0.5,dtype=int)
        img_j=np.array(img_j>0.5,dtype=int)
        for j in range(img1.shape[2]):
            nums.append([np.max(img1[:,:,j]),np.sum(img[:,:,j]),np.sum(img_i[:,:,j]),np.sum(img_j[:,:,j])])
            if np.max(img1[:,:,j])>0.5:
                res_seg1.append(evaluate_segmentation(img[:,:,j],img1[:,:,j],voxel_size))
                res_seg2.append(evaluate_segmentation(img_i[:,:,j],img1[:,:,j],voxel_size))
                res_seg3.append(evaluate_segmentation(img_j[:,:,j],img1[:,:,j],voxel_size))
        print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43


In [20]:
res_cls[res_cls[:,0]==1,4]

array([0., 0., 0., ..., 0., 0., 0.])

In [34]:
res_cls=np.array(res_cls)
eval=np.zeros((5,4))
for k in range(5):
    y1=res_cls[res_cls[:,0]==k,3]
    y=res_cls[res_cls[:,0]==k,6]
    eval[k,0:4]=compute_evaluation_metrics(y1,y)

In [35]:
print(np.round(np.mean(eval[:,0])*100,1),'pmin',np.round(np.std(eval[:,0])*100,1),'&',np.round(np.mean(eval[:,1])*100,1),'pmin',np.round(np.std(eval[:,1])*100,1),'&',np.round(np.mean(eval[:,2])*100,1),'pmin',np.round(np.std(eval[:,2])*100,1),'&',np.round(np.mean(eval[:,3])*100,1),'pmin',np.round(np.std(eval[:,3])*100,1))

92.6 pmin 1.6 & 74.1 pmin 1.9 & 95.1 pmin 1.7 & 68.1 pmin 7.4


In [38]:
y1=res_cls[:,3]
y=res_cls[:,6]
tp=np.sum(np.minimum(y1,y))
fn=np.sum(y1-np.minimum(y1,y))
fp=np.sum(y-np.minimum(y1,y))
tn=np.sum(np.ones(y.shape)-np.maximum(y1,y))
print(tp,fn,fp,tn)

852.0 298.0 409.0 8064.0


In [76]:
res_cls=np.array(res_cls)
nums=np.array(nums)
eval=np.zeros((5,3))
for k in range(5):
    y1=nums[res_cls[:,0]==k,0]
    y=nums[res_cls[:,0]==k,1]
    fpr,tpr,thresholds=roc_curve(y1,y)
    eval[k,0]=auc(fpr,tpr)

In [77]:
print(np.round(np.mean(eval[:,0])*100,1),'pmin',np.round(np.std(eval[:,0])*100,1))

85.2 pmin 3.0


In [47]:
con=np.zeros((2,2))
for i in range(res_cls.shape[0]):
    con[int(res_cls[i,3]==res_cls[i,5]),int(res_cls[i,3]==res_cls[i,6])]+=1
print(con)
print(mcnemar(con))

[[ 515.  172.]
 [ 192. 8744.]]
pvalue      0.3193143349234729
statistic   172.0


In [12]:
res_seg1=np.array(res_seg1)
res_seg2=np.array(res_seg2)
res_seg3=np.array(res_seg3)

In [60]:
i=5
print(np.round(np.nanmean(res_seg1[:,i])*100,1),'pmin',np.round(np.nanstd(res_seg1[:,i])*100,1),'&',np.round(np.nanmean(res_seg2[:,i])*100,1),'pmin',np.round(np.nanstd(res_seg2[:,i])*100,1),'&',np.round(np.nanmean(res_seg3[:,i])*100,1),'pmin',np.round(np.nanstd(res_seg3[:,i])*100,1))

82.9 pmin 20.9 & 83.1 pmin 20.3 & 85.4 pmin 19.3


In [56]:
print(np.round(np.nanmean(res_seg1[:,i]),1),'pmin',np.round(np.nanstd(res_seg1[:,i]),1),'&',np.round(np.nanmean(res_seg2[:,i]),1),'pmin',np.round(np.nanstd(res_seg2[:,i]),1),'&',np.round(np.nanmean(res_seg3[:,i]),1),'pmin',np.round(np.nanstd(res_seg3[:,i]),1))

11.8 pmin 17.6 & 11.3 pmin 18.6 & 10.7 pmin 17.8


In [67]:
for i in range(6):
    print(wilcoxon(res_seg2[:,i],res_seg3[:,i]))

WilcoxonResult(statistic=193212.0, pvalue=0.3193636583609757)
WilcoxonResult(statistic=191765.0, pvalue=0.23700543616339986)
WilcoxonResult(statistic=nan, pvalue=nan)
WilcoxonResult(statistic=156404.0, pvalue=0.012879992273866459)
WilcoxonResult(statistic=111800.5, pvalue=2.2094664063158117e-09)
WilcoxonResult(statistic=nan, pvalue=nan)


In [13]:
i=2
x=res_seg2[:,i]
y=res_seg3[:,i]
l=(np.isnan(x) | np.isnan(y))
wilcoxon(x[l==0],y[l==0])

WilcoxonResult(statistic=138210.5, pvalue=0.29200705399025495)

In [ ]:
compute_hd95_2d(img,img1,voxel_size)

In [30]:
print(files[i][:-4])
j=0
print(df1['case_id'][j])

case_0015
case_0105


In [18]:
df=pd.read_excel('C:/Users/Oona/Documents/Tpc/hnc/hnc_types.xlsx')

In [19]:
types=np.unique(list(df['type']))
numbers=np.zeros(len(types))
for i in range(df.shape[0]):
    numbers[np.where(types==df['type'][i])[0]]+=1
print(types)
print(numbers)

['hypopharynx' 'larynx' 'nan' 'nasopharynx' 'non-scc' 'oral' 'oropharynx'
 'other scc' 'unknown primary']
[12. 19.  0. 11. 19. 37. 53.  8. 17.]


In [38]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
#print(df1)
indices=[]
neg_indices=[]
for i in range(df1.shape[0]):
    if df1['label'][i]=='pos':
        if 'anon' in df1['anom_folder'][i]:
            indices.append(df1['anom_folder'][i][:-5])
        else:
            indices.append(df1['anom_folder'][i])
    if df1['label'][i]=='neg':
        neg_indices.append(df1['anom_folder'][i])
#print(indices)
for i in range(len(indices)):
    l=0
    for j in range(df.shape[0]):
        if int(indices[i])==df['index'][j]:
            l=1
    if l==0:
        print(indices[i])
print(neg_indices)
for i in range(len(neg_indices)):
    l=0
    for j in range(df.shape[0]):
        if isinstance(df['index'][j],int)==False:
            if neg_indices[i]==df['index'][j][1:]:
                l=1
    if l==0:
        print(neg_indices[i])

['1', '2', '3', '4', '5', '6', '7', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '47', '48', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122']
1
2
3
4
5
6
7
9
10
11
12
13
14
15
16
17
18
19
20
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
47
48


In [93]:
files[15]

'case_0082.nii'

In [36]:
df1=pd.read_csv('C:/Users/Oona/Downloads/handoff_oona/handoff_manifest.csv')
for k in range(1,2):
    folder_path='C:/Users/Oona/Downloads/handoff_oona/D006_PET/fold_{}'.format(k)
    files=os.listdir(folder_path)
    for i in range(15,16):#
        file_path='{}/{}'.format(folder_path,files[i])
        #img=np.array(nib.load(file_path).get_fdata()>0.5,dtype=int)
        #mask1_path='C:/Users/Oona/Downloads/handoff_oona/D007_PETMRI/fold_{}/{}'.format(k,files[i])
        #img_j=np.array(nib.load('C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])).get_fdata()>0.5,dtype=int)
        mask1_path='C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_{}/{}'.format(k,files[i])
        for j in range(df1.shape[0]):
            if df1['case_id'][j]==files[i][:-4]:
                if df1['label'][j]=='pos':
                    img1_folder='D:/img/hnc/anom_data/positiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    mask_path='{}/{}'.format(img1_folder1,img_files[j1])
                        if 'mri' in files1[i1] or 'MRI' in files1[i1] or 'spir' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                        if 'pet' in files1[i1] or 'PET' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img_path='{}/{}'.format(img1_folder1,img_files[j1])
                else:
                    print('neg')
                    img1_folder='D:/img/hnc/anom_data/negatiiviset/{}'.format(df1['anom_folder'][j])
                    files1=os.listdir(img1_folder)
                    for i1 in range(len(files1)):
                        if 'mask' in files1[i1] or 'Mask' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    mask_path='{}/{}'.format(img1_folder1,img_files[j1])
                        if 'mri' in files1[i1] or 'MRI' in files1[i1] or 'spir' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img1_path='{}/{}'.format(img1_folder1,img_files[j1])
                        if 'pet' in files1[i1] or 'PET' in files1[i1]:
                            img1_folder1='{}/{}'.format(img1_folder,files1[i1])
                            img_files=os.listdir(img1_folder1)
                            for j1 in range(len(img_files)):
                                if '.img' in img_files[j1]:
                                    img_path='{}/{}'.format(img1_folder1,img_files[j1])
        mask1=np.array(nib.load(mask1_path).get_fdata()>0.5,dtype=int)
        print(np.sum(mask1))
print(mask_path)
print(mask1_path)
print(img1_path)
print(img_path)

32678
D:/img/hnc/anom_data/positiiviset/7_anon/7_maski/7_maski.img
C:/Users/Oona/Downloads/handoff_oona/D008_PETMRIcond/fold_1/case_0082.nii
D:/img/hnc/anom_data/positiiviset/7_anon/7_nifti_mri/7_nifti_mri.img
D:/img/hnc/anom_data/positiiviset/7_anon/7_nifti_pet/7_nifti_pet.img


In [37]:
mask1=np.array(nib.load(mask1_path).get_fdata()>0.5,dtype=int)
mask=np.array(nib.load(mask_path).get_fdata()>0.5,dtype=int)
img=nib.load(img_path).get_fdata()
img1=nib.load(img1_path).get_fdata()
i=np.argmax(np.sum(np.sum(mask,axis=0),axis=0))

In [16]:
mask=np.zeros(img.shape)
print(np.sum(mask1))
i=np.argmax(np.sum(np.sum(mask1,axis=0),axis=0))

698


In [ ]:
print(evaluate_segmentation(mask1[:,:,i],mask[:,:,i],[1,1,1]))
print(evaluate_segmentation(mask1,mask,[1,1,1]))
plt.imshow(blue_col_img_1(np.flip(np.transpose(img1[:,:,i]),axis=0),np.flip(np.transpose(img[:,:,i]),axis=0)))
plot_outlines(np.flip(np.transpose(mask[:,:,i]),axis=0).T,color='skyblue')
plot_outlines(np.flip(np.transpose(mask1[:,:,i]),axis=0).T,color='white')
plt.axis('off')
fig=plt.gcf()
fig.savefig('fig_1_15.png',bbox_inches='tight')

In [ ]:
#0,0: 97.7
#1,15: 97.8
#0,6: 97.9

#0,3 34.5
#0.7 56.6
#0.11 0
#0.18 66.4
#0.38 0
#0.31 0
#0,43 0


#0,0: Dice: 96.6, 63.0
#2,5: 98.1, 97.6
#1,15: 97.7, 95.3
#0,6: 96.2, 93.9

#0,18: 67.3,65.8
#0,7: 60.2, 64.7
#0,3: 18.3, 43.9
#0.5: 0, 0
#0.11: 0,0
#0,34: 0,0
#0,38: 0,0

In [ ]:
b,a,cor_value,p_value,std_err=linregress(volumes[0,:],volumes[3,:])
print('Slope:',b)
print('Intercept:',a)
plt.scatter(volumes[0,:],volumes[3,:],color='blue')
plt.plot(np.array([-5,155]),b*np.array([-5,155])+a, color='black')
plt.axis('scaled')
plt.xlim([-5,155])
plt.ylim([-5,155])
plt.ylabel('Predicted GTV (cm3)',fontsize=14)
plt.xlabel('True GTV (cm3)',fontsize=14)
plt.title('GTV predicted via PET/MRI-SC',fontsize=15)
plt.rc('ytick',labelsize=15)
plt.rc('xtick',labelsize=14)
fig=plt.gcf()
fig.savefig('fig_6.png',bbox_inches='tight')